In [1]:
from pathlib import Path
import os
import gc
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# =========================================================
# 0. 경로 / 설정 (여기만 수정)
# =========================================================
# step1에서 만든 csv
STEP1_CSV = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step1_core_geometry/step1_core_geometry.csv")

# 원본 이미지 루트 (step1과 동일)
IMG_ROOT = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/1. 원본상추 crops_filter(1)")

# 출력 루트
OUT_ROOT = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step2_roi_crop")

# 어떤 샘플을 crop할지
USE_ONLY_ELLIPSE_OK = True
LIMIT_ROWS = None         # 테스트용 100 등, 전체는 None
MAX_WORKERS = 8
CHUNK_SIZE = 200

# crop 방식
# 1) ellipse bbox 기반으로 자르기
# 2) margin 추가
MARGIN_RATIO = 0.20       # major/minor axis 기준 여유분
MIN_MARGIN_PX = 10

# 결과 이미지 크기
SAVE_SQUARE_CROP = True
OUTPUT_SIZE = 256         # CNN 입력용 정사각형 저장

# 저장 옵션
SAVE_ROI_CROP = True
SAVE_ROI_MASK = True
SAVE_ROI_APPLIED = True   # 바깥 검정 처리된 버전
SAVE_OVERLAY = True
SAVE_EVERY_CHUNK = True

CSV_NAME = "step2_roi_crop.csv"
LOG_NAME = "step2_roi_crop_log.txt"

In [2]:


# =========================================================
# 1. 유틸
# =========================================================
def now_str():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def format_seconds(sec):
    sec = max(0, int(sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def write_log(msg: str):
    print(msg)
    with open(OUT_ROOT / LOG_NAME, "a", encoding="utf-8") as f:
        f.write(msg + "\n")


def safe_relpath(path: Path, root: Path):
    try:
        return str(path.relative_to(root))
    except Exception:
        return path.name


def clamp(v, low, high):
    return max(low, min(high, v))


def chunk_list(lst, chunk_size):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]


def resize_with_padding(img, out_size=256, is_mask=False):
    h, w = img.shape[:2]
    if h == 0 or w == 0:
        raise ValueError("empty image")

    scale = min(out_size / w, out_size / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    resized = cv2.resize(img, (new_w, new_h), interpolation=interp)

    if img.ndim == 2:
        canvas = np.zeros((out_size, out_size), dtype=img.dtype)
    else:
        canvas = np.zeros((out_size, out_size, img.shape[2]), dtype=img.dtype)

    x0 = (out_size - new_w) // 2
    y0 = (out_size - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = resized
    return canvas

# =========================================================
# 2. 타원 -> mask / crop box
# =========================================================
def build_ellipse_mask(h, w, ellipse_cx, ellipse_cy, major_axis, minor_axis, angle_deg):
    mask = np.zeros((h, w), dtype=np.uint8)

    center = (int(round(ellipse_cx)), int(round(ellipse_cy)))
    axes = (
        max(1, int(round(major_axis / 2.0))),
        max(1, int(round(minor_axis / 2.0))),
    )

    cv2.ellipse(
        mask,
        center=center,
        axes=axes,
        angle=float(angle_deg),
        startAngle=0,
        endAngle=360,
        color=255,
        thickness=-1,
    )
    return mask


def get_crop_box_from_ellipse(img_w, img_h, ellipse_cx, ellipse_cy, major_axis, minor_axis, margin_ratio=0.2, min_margin_px=10):
    # 타원 회전까지 완전 엄밀하게 박스화하지는 않고,
    # 축 기반 bbox + margin 방식으로 안정적으로 자름
    half_w = major_axis / 2.0
    half_h = minor_axis / 2.0

    # angle과 관계없이 조금 넉넉하게 major를 기준으로 잡음
    margin = max(min_margin_px, int(round(max(major_axis, minor_axis) * margin_ratio)))

    x1 = int(round(ellipse_cx - half_w)) - margin
    y1 = int(round(ellipse_cy - half_h)) - margin
    x2 = int(round(ellipse_cx + half_w)) + margin
    y2 = int(round(ellipse_cy + half_h)) + margin

    x1 = clamp(x1, 0, img_w - 1)
    y1 = clamp(y1, 0, img_h - 1)
    x2 = clamp(x2, 1, img_w)
    y2 = clamp(y2, 1, img_h)

    if x2 <= x1:
        x2 = min(img_w, x1 + 1)
    if y2 <= y1:
        y2 = min(img_h, y1 + 1)

    return x1, y1, x2, y2, margin

# =========================================================
# 3. 시각화
# =========================================================
def draw_overlay(img_bgr, row):
    vis = img_bgr.copy()

    center = (int(row["center_x"]), int(row["center_y"]))
    cv2.circle(vis, center, 5, (0, 140, 255), -1)  # 주황 중심점

    ellipse = (
        (float(row["ellipse_cx"]), float(row["ellipse_cy"])),
        (float(row["major_axis"]), float(row["minor_axis"])),
        float(row["angle_deg"]),
    )
    cv2.ellipse(vis, ellipse, (255, 0, 0), 2)      # 파란 타원

    x1 = int(row["crop_x1"])
    y1 = int(row["crop_y1"])
    x2 = int(row["crop_x2"])
    y2 = int(row["crop_y2"])
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)  # 초록 crop box

    return vis

# =========================================================
# 4. 단일 행 처리
# =========================================================
def process_one_row(row_dict):
    try:
        img_path = Path(row_dict["image_path"])
        rel_path = row_dict.get("rel_path", img_path.name)

        img = cv2.imread(str(img_path))
        if img is None:
            return {
                "image_path": str(img_path),
                "rel_path": rel_path,
                "status": "fail",
                "fail_reason": "imread_failed",
            }

        h, w = img.shape[:2]

        ellipse_cx = float(row_dict["ellipse_cx"])
        ellipse_cy = float(row_dict["ellipse_cy"])
        major_axis = float(row_dict["major_axis"])
        minor_axis = float(row_dict["minor_axis"])
        angle_deg = float(row_dict["angle_deg"])

        if np.isnan(ellipse_cx) or np.isnan(ellipse_cy) or np.isnan(major_axis) or np.isnan(minor_axis):
            return {
                "image_path": str(img_path),
                "rel_path": rel_path,
                "status": "fail",
                "fail_reason": "ellipse_nan",
            }

        ellipse_mask = build_ellipse_mask(h, w, ellipse_cx, ellipse_cy, major_axis, minor_axis, angle_deg)
        x1, y1, x2, y2, margin = get_crop_box_from_ellipse(
            img_w=w,
            img_h=h,
            ellipse_cx=ellipse_cx,
            ellipse_cy=ellipse_cy,
            major_axis=major_axis,
            minor_axis=minor_axis,
            margin_ratio=MARGIN_RATIO,
            min_margin_px=MIN_MARGIN_PX,
        )

        crop_img = img[y1:y2, x1:x2].copy()
        crop_mask = ellipse_mask[y1:y2, x1:x2].copy()

        roi_applied = cv2.bitwise_and(crop_img, crop_img, mask=crop_mask)

        # 저장 경로
        rel_no_ext = Path(rel_path).with_suffix("")
        crop_path = OUT_ROOT / "roi_crop"
        mask_path = OUT_ROOT / "roi_mask"
        applied_path = OUT_ROOT / "roi_applied"
        overlay_path = OUT_ROOT / "overlay"

        crop_save = crop_path / rel_no_ext.with_suffix(".png")
        mask_save = mask_path / rel_no_ext.with_suffix(".png")
        applied_save = applied_path / rel_no_ext.with_suffix(".png")
        overlay_save = overlay_path / rel_no_ext.with_suffix(".png")

        ensure_dir(crop_save.parent)
        ensure_dir(mask_save.parent)
        ensure_dir(applied_save.parent)
        ensure_dir(overlay_save.parent)

        save_crop_img = resize_with_padding(crop_img, OUTPUT_SIZE, is_mask=False) if SAVE_SQUARE_CROP else crop_img
        save_mask_img = resize_with_padding(crop_mask, OUTPUT_SIZE, is_mask=True) if SAVE_SQUARE_CROP else crop_mask
        save_applied_img = resize_with_padding(roi_applied, OUTPUT_SIZE, is_mask=False) if SAVE_SQUARE_CROP else roi_applied

        if SAVE_ROI_CROP:
            cv2.imwrite(str(crop_save), save_crop_img)
        if SAVE_ROI_MASK:
            cv2.imwrite(str(mask_save), save_mask_img)
        if SAVE_ROI_APPLIED:
            cv2.imwrite(str(applied_save), save_applied_img)
        if SAVE_OVERLAY:
            row_for_overlay = dict(row_dict)
            row_for_overlay.update({
                "crop_x1": x1,
                "crop_y1": y1,
                "crop_x2": x2,
                "crop_y2": y2,
            })
            overlay = draw_overlay(img, row_for_overlay)
            cv2.imwrite(str(overlay_save), overlay)

        out = {
            "image_path": str(img_path),
            "rel_path": rel_path,
            "image_name": row_dict.get("image_name", img_path.name),
            "stem": row_dict.get("stem", img_path.stem),
            "status": "ok",
            "fail_reason": "",
            "center_x": row_dict.get("center_x", np.nan),
            "center_y": row_dict.get("center_y", np.nan),
            "ellipse_cx": ellipse_cx,
            "ellipse_cy": ellipse_cy,
            "major_axis": major_axis,
            "minor_axis": minor_axis,
            "angle_deg": angle_deg,
            "crop_x1": x1,
            "crop_y1": y1,
            "crop_x2": x2,
            "crop_y2": y2,
            "crop_w": int(x2 - x1),
            "crop_h": int(y2 - y1),
            "margin_px": int(margin),
            "roi_crop_path": safe_relpath(crop_save, OUT_ROOT) if SAVE_ROI_CROP else None,
            "roi_mask_path": safe_relpath(mask_save, OUT_ROOT) if SAVE_ROI_MASK else None,
            "roi_applied_path": safe_relpath(applied_save, OUT_ROOT) if SAVE_ROI_APPLIED else None,
            "overlay_path": safe_relpath(overlay_save, OUT_ROOT) if SAVE_OVERLAY else None,
            "output_size": OUTPUT_SIZE if SAVE_SQUARE_CROP else np.nan,
        }
        return out

    except Exception as e:
        return {
            "image_path": row_dict.get("image_path", ""),
            "rel_path": row_dict.get("rel_path", ""),
            "status": "fail",
            "fail_reason": f"exception: {type(e).__name__}: {e}",
        }

# =========================================================
# 5. chunk 저장
# =========================================================
def save_chunk_rows(rows, chunk_idx):
    if not rows:
        return None
    chunk_dir = OUT_ROOT / "chunks"
    ensure_dir(chunk_dir)
    chunk_path = chunk_dir / f"chunk_{chunk_idx:03d}.csv"
    pd.DataFrame(rows).to_csv(chunk_path, index=False, encoding="utf-8-sig")
    return chunk_path



In [3]:
# =========================================================
# 6. 메인
# =========================================================
def main():
    ensure_dir(OUT_ROOT)
    ensure_dir(OUT_ROOT / "roi_crop")
    ensure_dir(OUT_ROOT / "roi_mask")
    ensure_dir(OUT_ROOT / "roi_applied")
    ensure_dir(OUT_ROOT / "overlay")
    ensure_dir(OUT_ROOT / "chunks")

    with open(OUT_ROOT / LOG_NAME, "w", encoding="utf-8") as f:
        f.write("")

    start_time = time.time()
    write_log("=" * 90)
    write_log(f"[START] {now_str()}")
    write_log(f"STEP1_CSV     : {STEP1_CSV}")
    write_log(f"IMG_ROOT      : {IMG_ROOT}")
    write_log(f"OUT_ROOT      : {OUT_ROOT}")
    write_log(f"MAX_WORKERS   : {MAX_WORKERS}")
    write_log(f"CHUNK_SIZE    : {CHUNK_SIZE}")
    write_log(f"OUTPUT_SIZE   : {OUTPUT_SIZE} (square={SAVE_SQUARE_CROP})")

    df = pd.read_csv(STEP1_CSV)

    if USE_ONLY_ELLIPSE_OK and "ellipse_ok" in df.columns:
        df = df[df["ellipse_ok"] == 1].copy()

    df = df.dropna(subset=["image_path", "ellipse_cx", "ellipse_cy", "major_axis", "minor_axis", "angle_deg"]).copy()
    df = df.reset_index(drop=True)

    if LIMIT_ROWS is not None:
        df = df.iloc[:LIMIT_ROWS].copy()

    total_n = len(df)
    write_log(f"TOTAL_ROWS    : {total_n}")

    if total_n == 0:
        write_log("[STOP] 처리할 행이 없습니다.")
        return

    records = df.to_dict(orient="records")

    all_rows = []
    total_done = 0
    chunk_idx = 0

    for rows_chunk in chunk_list(records, CHUNK_SIZE):
        chunk_idx += 1
        chunk_start = time.time()
        rows_out = []

        write_log("-" * 90)
        write_log(f"[CHUNK {chunk_idx}] start / size={len(rows_chunk)} / done={total_done}/{total_n}")

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {ex.submit(process_one_row, r): r for r in rows_chunk}
            for fut in tqdm(as_completed(futures), total=len(futures), desc=f"chunk {chunk_idx}"):
                row = fut.result()
                rows_out.append(row)
                total_done += 1

        rows_df = pd.DataFrame(rows_out)
        if len(rows_df) > 0:
            rows_df = rows_df.sort_values("rel_path").reset_index(drop=True)
            rows_list = rows_df.to_dict(orient="records")
        else:
            rows_list = []

        all_rows.extend(rows_list)

        if SAVE_EVERY_CHUNK:
            chunk_path = save_chunk_rows(rows_list, chunk_idx)
            write_log(f"[CHUNK {chunk_idx}] saved: {chunk_path}")

        ok_n = int((rows_df["status"] == "ok").sum()) if "status" in rows_df.columns else 0
        fail_n = int((rows_df["status"] == "fail").sum()) if "status" in rows_df.columns else 0
        chunk_elapsed = time.time() - chunk_start
        total_elapsed = time.time() - start_time
        eta = (total_elapsed / total_done) * (total_n - total_done) if total_done > 0 else 0

        write_log(
            f"[CHUNK {chunk_idx}] ok={ok_n}, fail={fail_n}, "
            f"chunk_time={format_seconds(chunk_elapsed)}, total_done={total_done}/{total_n}, ETA={format_seconds(eta)}"
        )

        del rows_df, rows_list
        gc.collect()

    final_df = pd.DataFrame(all_rows)
    final_df = final_df.sort_values("rel_path").reset_index(drop=True)
    final_csv_path = OUT_ROOT / CSV_NAME
    final_df.to_csv(final_csv_path, index=False, encoding="utf-8-sig")

    fail_summary_path = OUT_ROOT / "fail_reason_summary.csv"
    final_df["fail_reason"].fillna("ok").value_counts().rename_axis("fail_reason").reset_index(name="count").to_csv(
        fail_summary_path, index=False, encoding="utf-8-sig"
    )

    ok_n = int((final_df["status"] == "ok").sum()) if "status" in final_df.columns else 0
    fail_n = int((final_df["status"] == "fail").sum()) if "status" in final_df.columns else 0

    write_log("=" * 90)
    write_log(f"[FINAL CSV] {final_csv_path}")
    write_log(f"[FAIL SUMMARY] {fail_summary_path}")
    write_log(f"[RESULT] ok={ok_n}, fail={fail_n}, total={len(final_df)}")
    write_log(f"[END] {now_str()} / elapsed={format_seconds(time.time() - start_time)}")
    write_log("=" * 90)



In [4]:

if __name__ == "__main__":
    main()


[START] 2026-03-29 07:32:09
STEP1_CSV     : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step1_core_geometry/step1_core_geometry.csv
IMG_ROOT      : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/1. 원본상추 crops_filter(1)
OUT_ROOT      : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step2_roi_crop
MAX_WORKERS   : 8
CHUNK_SIZE    : 200
OUTPUT_SIZE   : 256 (square=True)
TOTAL_ROWS    : 1088
------------------------------------------------------------------------------------------
[CHUNK 1] start / size=200 / done=0/1088


chunk 1:   0%|          | 0/200 [00:00<?, ?it/s]

[CHUNK 1] saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step2_roi_crop/chunks/chunk_001.csv
[CHUNK 1] ok=200, fail=0, chunk_time=00:00:27, total_done=200/1088, ETA=00:02:04
------------------------------------------------------------------------------------------
[CHUNK 2] start / size=200 / done=200/1088


chunk 2:   0%|          | 0/200 [00:00<?, ?it/s]

[CHUNK 2] saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step2_roi_crop/chunks/chunk_002.csv
[CHUNK 2] ok=200, fail=0, chunk_time=00:00:25, total_done=400/1088, ETA=00:01:31
------------------------------------------------------------------------------------------
[CHUNK 3] start / size=200 / done=400/1088


...

[CHUNK 4] saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step2_roi_crop/chunks/chunk_004.csv
[CHUNK 4] ok=200, fail=0, chunk_time=00:00:26, total_done=800/1088, ETA=00:00:38
------------------------------------------------------------------------------------------
[CHUNK 5] start / size=200 / done=800/1088


chunk 5:   0%|          | 0/200 [00:00<?, ?it/s]

[CHUNK 5] saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step2_roi_crop/chunks/chunk_005.csv
[CHUNK 5] ok=200, fail=0, chunk_time=00:00:25, total_done=1000/1088, ETA=00:00:11
------------------------------------------------------------------------------------------
[CHUNK 6] start / size=88 / done=1000/1088


chunk 6:   0%|          | 0/88 [00:00<?, ?it/s]

[CHUNK 6] saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step2_roi_crop/chunks/chunk_006.csv
[CHUNK 6] ok=88, fail=0, chunk_time=00:00:11, total_done=1088/1088, ETA=00:00:00
[FINAL CSV] /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step2_roi_crop/step2_roi_crop.csv
[FAIL SUMMARY] /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/5. 추출+CNN Pipeline/step2_roi_crop/fail_reason_summary.csv
[RESULT] ok=1088, fail=0, total=1088
[END] 2026-03-29 07:34:34 / elapsed=00:02:24
